# 🔬 Phase 3B — Proper Validation-Based Temperature Selection

**Methodological Protocol:**  
$$\text{TRAIN } (O_{\text{train}}=48) \longrightarrow \text{VAL } (O=48, \text{ select } \tau) \longrightarrow \text{TEST } (\text{Locked multi-horizon evaluation})$$

> **Why this notebook exists:**  
> In Phase 2, temperature scaling was explored across all variants directly on test MSE.  
> To make this research scientifically bulletproof for your final-year thesis defense, this notebook enforces **strict zero-lookahead validation selection**:
> 1. Loads or trains the candidate models ($\tau \in \{0.5, 1.0, 2.0, 4.0\}$)
> 2. Inspects **only** validation loss at $O=48$ to pick the best $\tau$ for each (dataset, seed)
> 3. Evaluates **only** the selected model on the locked test set across horizons 24 to 720
> 4. Generates the final validated comparison table against the baseline.

In [ ]:
# Phase 3B — Validation-Based Temperature Selection
# METHODOLOGY: TRAIN -> VAL (select tau) -> TEST (locked, reported once)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os, sys

# Auto-detect project folder in Google Drive (handles casing and paths)
candidate_paths = [
    '/content/drive/MyDrive/D2Vformer',
    '/content/drive/MyDrive/D2vformer',
    '/content/drive/MyDrive/d2vformer',
    '/content/drive/MyDrive/AYUSH PROGRAMMING/D2vformer',
    '/content/drive/MyDrive/AYUSH PROGRAMMING/D2Vformer',
]
PROJECT_ROOT = None
for p in candidate_paths:
    if os.path.isdir(p):
        PROJECT_ROOT = p
        break

if PROJECT_ROOT is None:
    # Search top-level folders on MyDrive
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if 'models' in dirs and 'experiments' in dirs:
            PROJECT_ROOT = root
            break

if PROJECT_ROOT is None or not os.path.isdir(PROJECT_ROOT):
    raise FileNotFoundError("Could not auto-locate D2Vformer project folder in Google Drive. Please set PROJECT_ROOT manually.")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'Project root : {PROJECT_ROOT}')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device       : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU          : {torch.cuda.get_device_name(0)}')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scipy', 'pandas', 'numpy', 'matplotlib', 'tqdm'], check=True)

from models.temperature_d2vformer import TemperaturePureD2Vformer
from experiments.temperature_experiment import train_temperature_d2vformer
from utils.data import get_data_loaders
from utils.reproducibility import set_seed, compute_parameter_checksum
import numpy as np, pandas as pd, math, time

print('Imports and modules loaded successfully.')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3B — STEP 1: Verify / Train Checkpoints & Extract Validation Loss
# If checkpoints are already in Drive, loads them instantly.
# If missing, trains them automatically on GPU (~1-2 min total).
# ─────────────────────────────────────────────────────────────────────────────
DATASETS       = ['ETTh1', 'exchange']
SEEDS          = [42, 43, 44]
TAU_CANDIDATES = [0.5, 1.0, 2.0, 4.0]   # Candidate set for validation selection
EVAL_HORIZONS  = [24, 48, 96, 192, 336, 720]
CKPT_DIR       = os.path.join(PROJECT_ROOT, 'results', 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

print('Verifying checkpoints in results/checkpoints/ ...')
print('=' * 65)

val_loss_records = []

for dataset in DATASETS:
    for seed in SEEDS:
        print(f'\n[{dataset} | seed={seed}]')
        for tau in TAU_CANDIDATES:
            tag       = f'tau{tau}'
            ckpt_fn   = f'temp_d2v_{dataset}_{tag}_seed{seed}.pt'
            ckpt_path = os.path.join(CKPT_DIR, ckpt_fn)
            
            # Train if missing
            if not os.path.exists(ckpt_path):
                print(f'  Training missing: {ckpt_fn} ...')
                t0 = time.time()
                train_temperature_d2vformer(
                    dataset_name=dataset, seq_len=96, train_horizon=48,
                    d_model=128, d_ff=256, k_freq=16, dropout=0.05,
                    temperature_mode='fixed', initial_temperature=tau,
                    lr=1e-3, epochs=10, patience=3, batch_size=64,
                    seed=seed, device=DEVICE, checkpoint_dir=CKPT_DIR
                )
                print(f'    Done training in {time.time()-t0:.1f}s')
            
            # Load checkpoint & extract val_loss
            ckpt = torch.load(ckpt_path, map_location='cpu')
            val_loss = ckpt.get('val_loss', None)
            best_ep  = ckpt.get('best_epoch', None)
            
            print(f'  tau={tau:3.1f}: val_loss={val_loss:.6f}  (best_epoch={best_ep})')
            val_loss_records.append({
                'dataset': dataset, 'seed': seed, 'tau': tau,
                'val_loss': val_loss, 'best_epoch': best_ep,
                'ckpt_path': ckpt_path
            })

print('=' * 65)
print(f'All {len(val_loss_records)} checkpoints ready.')

df_val = pd.DataFrame(val_loss_records)
print('\nValidation loss table:')
print(df_val[['dataset', 'seed', 'tau', 'val_loss', 'best_epoch']].to_string(index=False))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3B — STEP 2: Select tau per (dataset, seed) using ONLY val_loss
# STRICT HYPERPARAMETER SELECTION: TEST SET IS NOT TOUCHED!
# ─────────────────────────────────────────────────────────────────────────────
selection_records = []

print('Tau selection based exclusively on VALIDATION MSE (test set NOT touched):')
print('=' * 65)

for dataset in DATASETS:
    for seed in SEEDS:
        sub = df_val[(df_val['dataset'] == dataset) & (df_val['seed'] == seed)]
        best_row = sub.loc[sub['val_loss'].idxmin()]
        tau_sel  = best_row['tau']
        val_sel  = best_row['val_loss']
        
        print(f'\n[{dataset} | seed={seed}]')
        for _, row in sub.iterrows():
            marker = '  <-- SELECTED (min val loss)' if row['tau'] == tau_sel else ''
            print(f'  tau={row["tau"]:3.1f}: val_loss={row["val_loss"]:.6f}{marker}')
        
        selection_records.append({
            'dataset': dataset, 'seed': seed,
            'selected_tau': tau_sel,
            'val_loss_selected': val_sel,
            'ckpt_path': best_row['ckpt_path']
        })

df_sel = pd.DataFrame(selection_records)
print('\n' + '=' * 65)
print('VALIDATION SELECTION DECISIONS:')
print('=' * 65)
print(df_sel[['dataset', 'seed', 'selected_tau', 'val_loss_selected']].to_string(index=False))


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3B — STEP 3: Evaluate ONLY Selected tau on the LOCKED Test Set
# Evaluates selected model across horizons O in [24, 48, 96, 192, 336, 720]
# Also evaluates baseline tau=1.0 for direct one-to-one comparison.
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_on_test(ckpt_path, eval_horizon, dataset_name, batch_size=64, device='cpu'):
    ckpt = torch.load(ckpt_path, map_location=device)
    model = TemperaturePureD2Vformer(
        c_in=ckpt['c_in'], seq_len=ckpt['seq_len'],
        d_model=ckpt['d_model'], d_ff=ckpt['d_ff'],
        k_freq=ckpt['k_freq'], dropout=ckpt['dropout'],
        temperature_mode=ckpt['temperature_mode'],
        initial_temperature=ckpt['initial_temperature']
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    
    # Checksum verification
    assert compute_parameter_checksum(model) == ckpt['checksum'], 'Checksum verification failed!'
    
    actual_bs = min(batch_size, 32 if eval_horizon >= 336 else batch_size)
    _, _, test_loader, _ = get_data_loaders(
        dataset_name=dataset_name, seq_len=ckpt['seq_len'],
        pred_len=eval_horizon, batch_size=actual_bs,
        data_root=os.path.join(PROJECT_ROOT, 'datasets')
    )
    
    sq_err, abs_err, n = 0.0, 0.0, 0
    with torch.no_grad():
        for bx, by, bxm, bym in test_loader:
            bx, by, bxm, bym = bx.to(device), by.to(device), bxm.to(device), bym.to(device)
            out, _ = model(bx, bxm, bym)
            diff = (out - by).cpu().numpy()
            sq_err  += float(np.sum(diff**2))
            abs_err += float(np.sum(np.abs(diff)))
            n += diff.size
    return {'mse': sq_err/n, 'mae': abs_err/n, 'tau': float(ckpt['final_temperature'])}

locked_records = []

print('EVALUATION ON LOCKED TEST SET (evaluated once per seed)')
print('=' * 65)

for _, sel_row in df_sel.iterrows():
    dataset    = sel_row['dataset']
    seed       = sel_row['seed']
    tau_sel    = sel_row['selected_tau']
    ckpt_path  = sel_row['ckpt_path']
    
    tau1_ckpt = os.path.join(CKPT_DIR, f'temp_d2v_{dataset}_tau1.0_seed{seed}.pt')
    
    print(f'\n[{dataset} | seed={seed} | Selected tau={tau_sel}]')
    
    for O in EVAL_HORIZONS:
        res_sel = evaluate_on_test(ckpt_path, O, dataset, device=DEVICE)
        res_t1  = evaluate_on_test(tau1_ckpt, O, dataset, device=DEVICE)
        
        rel_diff = (res_t1['mse'] - res_sel['mse']) / res_t1['mse'] * 100.0
        locked_records.append({
            'dataset': dataset, 'seed': seed, 'selected_tau': tau_sel,
            'eval_horizon': O,
            'mse_selected': round(res_sel['mse'], 5),
            'mae_selected': round(res_sel['mae'], 5),
            'mse_tau1_baseline': round(res_t1['mse'], 5),
            'mae_tau1_baseline': round(res_t1['mae'], 5),
            'rel_improvement_pct': round(rel_diff, 2)
        })
        print(f'  O={O:3d}: MSE(selected)={res_sel["mse"]:.4f}  MSE(tau=1)={res_t1["mse"]:.4f}  ({rel_diff:+.2f}%)')

df_locked = pd.DataFrame(locked_records)
out_csv = os.path.join(PROJECT_ROOT, 'results', 'temperature', 'final_locked_test_results.csv')
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
df_locked.to_csv(out_csv, index=False)
print(f'\nSaved locked test results to: {out_csv}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3B — STEP 4: Validation vs Test Comparison Tables
# ─────────────────────────────────────────────────────────────────────────────
print('=' * 75)
print('VALIDATION-CONFIRMED SUMMARY (Mean across 3 Seeds)')
print('=' * 75)

print('\n1. Mean across all horizons:')
summary_overall = df_locked.groupby('dataset')[['mse_selected', 'mse_tau1_baseline']].mean().round(4)
summary_overall['rel_improvement_pct'] = (
    (summary_overall['mse_tau1_baseline'] - summary_overall['mse_selected']) / summary_overall['mse_tau1_baseline'] * 100
).round(2)
print(summary_overall.to_string())

print('\n2. Per-horizon breakdown:')
summary_ph = df_locked.groupby(['dataset', 'eval_horizon'])[['mse_selected', 'mse_tau1_baseline']].mean().round(4)
summary_ph['rel_improvement_pct'] = (
    (summary_ph['mse_tau1_baseline'] - summary_ph['mse_selected']) / summary_ph['mse_tau1_baseline'] * 100
).round(2)
print(summary_ph.to_string())

# Save validation selection log
vs_records = []
for dataset in DATASETS:
    for seed in SEEDS:
        sub = df_val[(df_val['dataset'] == dataset) & (df_val['seed'] == seed)]
        sel_tau = df_sel[(df_sel['dataset'] == dataset) & (df_sel['seed'] == seed)]['selected_tau'].iloc[0]
        for _, row in sub.iterrows():
            vs_records.append({
                'dataset': dataset, 'seed': seed, 'tau_candidate': row['tau'],
                'val_loss': row['val_loss'], 'best_epoch': row['best_epoch'],
                'is_selected': bool(row['tau'] == sel_tau)
            })

vs_csv = os.path.join(PROJECT_ROOT, 'results', 'temperature', 'validation_selection.csv')
pd.DataFrame(vs_records).to_csv(vs_csv, index=False)
print(f'\nSaved validation selection details to: {vs_csv}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3B — STEP 5: Statistical Significance Analysis
# ─────────────────────────────────────────────────────────────────────────────
from scipy import stats

print('=' * 75)
print('PHASE 3B STATISTICAL ANALYSIS (Validation-Selected tau vs tau=1.0 Baseline)')
print('=' * 75)

for dataset in DATASETS:
    print(f'\nDataset: {dataset}')
    sub = df_locked[df_locked['dataset'] == dataset]
    for O in EVAL_HORIZONS:
        row = sub[sub['eval_horizon'] == O]
        sel_mses = row['mse_selected'].values
        t1_mses  = row['mse_tau1_baseline'].values
        diff     = sel_mses - t1_mses
        
        t_stat, p_val = stats.ttest_rel(t1_mses, sel_mses)
        mean_sel = sel_mses.mean()
        mean_t1  = t1_mses.mean()
        rel_imp  = (mean_t1 - mean_sel) / mean_t1 * 100.0
        all_better = (diff <= 0).all()
        
        print(f'  O={O:3d}: Selected={mean_sel:.4f} | Baseline={mean_t1:.4f} | RelImp={rel_imp:+.2f}% | p={p_val:.3f} | AllSeedsBetter={all_better}')

print('\nDone! Download the two CSV files from results/temperature/ to your local PC:')
print('  1. results/temperature/final_locked_test_results.csv')
print('  2. results/temperature/validation_selection.csv')
